In [1]:
import sys, os
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [2]:
%LoadCheckpoint /home/colinc/code/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten_cpu/o4_mini_high/checkpoints/post_cell_22_try_0.pickle

trying: ['title_for_x_axis', 'title_for_x_axis_cell29', 'title_for_x_axis_cell20', 'title_for_x_axis_cell22', 'title_for_x_axis_cell33']
me:  6
me:  17
me:  8
me:  10
me:  21
trying: ['title_for_x_axis', 'title_for_x_axis_cell29', 'title_for_x_axis_cell20', 'title_for_x_axis_cell22', 'title_for_x_axis_cell33']
me:  6
me:  17
me:  8
me:  10
me:  21
trying: ['responses_df_2022', 'responses_df_2022_cell10']
me:  16
me:  16
trying: ['responses_df_2022', 'responses_df_2022_cell10']
me:  16
me:  16
trying: ['title_for_x_axis', 'title_for_x_axis_cell29', 'title_for_x_axis_cell20', 'title_for_x_axis_cell22', 'title_for_x_axis_cell33']
me:  6
me:  17
me:  8
me:  10
me:  21
trying: ['title_for_x_axis', 'title_for_x_axis_cell29', 'title_for_x_axis_cell20', 'title_for_x_axis_cell22', 'title_for_x_axis_cell33']
me:  6
me:  17
me:  8
me:  10
me:  21
trying: ['question_of_interest_cell23']
me:  11
trying: ['count_then_return_percent_23']
me:  11
trying: ['percentages_cell23']
me:  11
trying: ['orient

me:  17
trying: ['df']
me:  14
trying: ['responses_df_2021']
me:  17
trying: ['count_then_return_percent_33']
me:  21
trying: ['px']
me:  0
trying: ['responses_df_2019_cell10']
me:  21
trying: ['warnings']
me:  0
trying: ['np']
me:  0
trying: ['responses_df_2020']


me:  21
trying: ['go']
me:  0
trying: ['subset_of_countries']
me:  6
trying: ['question_of_interest_cell33']
me:  21
trying: ['responses_df_2018_cell10']
me:  21
trying: ['country_df_combined']
me:  6
trying: ['title_for_x_axis']
me:  6
trying: ['combine_subset_of_data_from_multiple_years_33']
me:  21
trying: ['question_name_alternate']
me:  6
trying: ['programming_df_combined']
me:  21
trying: ['question_name_alternate_cell18']
me:  6
trying: ['count_then_return_percent_18']
me:  6
trying: ['add_year_column_to_dataframes_18']
me:  6
trying: ['combine_subset_of_data_from_multiple_years_18']
me:  6
trying: ['country_df_combined_cell18']
me:  6
trying: ['question_name']
me:  6
trying: ['question_of_interest_alternate']
me:  14
trying: ['question_of_interest_cell18']
me:  6
trying: ['question_of_interest_cell26']
me:  14
trying: ['grab_subset_of_data_26']
me:  14
trying: ['add_year_column_to_dataframes_26']
me:  14
trying: ['convert_df_of_counts_to_percentages_26']
me:  14
trying: ['learn

In [3]:
%%RecordEvent
%%time
### cell 23 ###

def grab_subset_of_data_35(df, question_of_interest):
    # Vectorized column filtering and renaming without an explicit full copy
    mask = df.columns.str.contains(question_of_interest)
    cols = df.columns[mask]
    # Derive new column names by splitting at the last '-'
    new_names = [c.rsplit('-', 1)[-1].lstrip() for c in cols]
    df_sub = df.loc[:, cols]
    df_sub.columns = new_names
    return df_sub.dropna(how='all', subset=new_names)


def combine_subset_of_data_from_multiple_years_for_multiple_choice_multiple_response_questions_35(question_of_interest, include_2017=None):
    # Define (dataframe, year) pairs in chronological order
    sources = [
        (responses_df_2018_cell10, '2018'),
        (responses_df_2019_cell10, '2019'),
        (responses_df_2020,        '2020'),
        (responses_df_2021,        '2021'),
        (responses_df_2022_cell10, '2022'),
    ]
    if include_2017 is not None:
        sources.insert(0, (responses_df_2017, '2017'))
    # Subset, rename, dropna, and assign year in one comprehension
    dfs = [
        grab_subset_of_data_35(src_df, question_of_interest).assign(year=yr)
        for src_df, yr in sources
    ]
    # Concatenate once with a fresh integer index
    df_combined = pd.concat(dfs, ignore_index=True)
    # Count non-null responses per year
    df_combined_counts = df_combined.groupby('year').count().reset_index()
    return df_combined, df_combined_counts


def convert_df_of_counts_to_percentages_35(df, df_counts):
    # Cast counts to int and set 'year' as index for alignment
    df_counts_int = df_counts.astype(int).set_index('year')
    # Compute total respondents per year once
    totals = df['year'].value_counts().reindex(df_counts_int.index)
    # Divide and multiply by 100, then bring 'year' back as a column
    percentages = df_counts_int.div(totals, axis=0).mul(100).reset_index()
    return percentages

# ---- Pipeline ----
question_of_interest_cell35 = 'What programming languages do you use on a regular basis?'

language_df_combined, language_df_combined_counts = \
    combine_subset_of_data_from_multiple_years_for_multiple_choice_multiple_response_questions_35(
        question_of_interest_cell35
    )

language_df_combined_percentages = \
    convert_df_of_counts_to_percentages_35(
        language_df_combined,
        language_df_combined_counts
    )

# Select the languages of interest and reshape
languages = ['Python', 'SQL', 'C++', 'C', 'R', 'Java', 'Javascript', 'Other']

df_cell35 = (
    language_df_combined_percentages
        .loc[:, ['year'] + languages]
        .melt(id_vars=['year'], value_vars=languages)
        .sort_values(by=['year', 'value'])
        .rename(columns={'variable': ''})
)

# Display info
df_cell35.info()

<class 'pandas.core.frame.DataFrame'>
Index: 40 entries, 0 to 39
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   year    40 non-null     int64  
 1           40 non-null     object 
 2   value   0 non-null      float64
dtypes: float64(1), int64(1), object(1)
memory usage: 1.2+ KB
CPU times: user 43.1 ms, sys: 3.86 ms, total: 47 ms
Wall time: 47 ms


In [4]:
%Checkpoint /home/colinc/code/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten_cpu/o4_mini_high/checkpoints/post_cell_23_try_1.pickle

migration speed (bps): 724826253.7428352


---------------------------
variables to migrate:
learning_platform_df_combined_percentages_cell26 849
learning_platform_df_combined_percentages 1089
question_of_interest_alternate 123
question_of_interest_cell23 116
count_then_return_percent_23 144
percentages_cell23 151
orientation_for_chart 50
India_responses_df_2022 9195445
bar_chart_multiple_choice_24 144
USA_responses_df_2022 3060721
age_df_combined_cell22 10611
count_then_return_percent_24 144
count_then_return_percent_22 144
percentages_cell24 151
question_of_interest_cell22 87
title_for_chart_cell24 106
combine_subset_of_data_from_multiple_years_22 144
title_for_y_axis_cell24 65
add_year_column_to_dataframes_22 144
question_of_interest_cell24 116
title_for_x_axis_cell22 49
language_df_combined_percentages 1144
replace_hyphen_with_en_dash 144
convert_df_of_counts_to_percentages_35 144
grab_subset_of_data_35 144
question_of_interest_cell35 106
df_cell35 3421
language_df_combined 8514579
language_df_combined_counts 1409
languages

In [5]:
%PrintCellInfo opt_cell_exec_info

======= Cell 0 =======
Input variables []
Active variables []
Intermediate variables []
Future variables []
Modified dataframes
======= Cell 1 =======
Input variables []
Active variables ['responses_df_2019', 'responses_df_2018', 'responses_df_2021', 'responses_df_2022', 'responses_df_2020']
Intermediate variables ['file_name_2019', 'base_dir_2018', 'file_name_2020', 'factor', 'base_dir_2021', 'base_dir_2020', 'base_dir_2019', 'directory', 'directory_cell8', 'file_name_2018', 'base_dir_2017', 'file_name_2021', 'file_name_2022', 'base_dir_2022', 'responses_df_2017', 'file_name_2017']
Future variables ['percentages', 'responses_df_2017', 'responses_df_2022_cell10']
Modified dataframes
======= Cell 2 =======
Input variables ['responses_df_2022', 'responses_df_2018', 'responses_df_2019']
Active variables ['responses_df_2022', 'responses_df_2019_cell10', 'responses_df_2022_cell10']
Intermediate variables ['responses_df_2018_cell10']
Future variables ['responses_df_2018_cell10', 'responses_d

In [6]:

with open("/home/colinc/code/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/opt_cell_exec_info_23_try_1.pkl", "wb") as f:
    pickle.dump(opt_cell_exec_info[23], f)


In [7]:
opt_output = Out.get(4)

In [8]:
%LoadCheckpoint /home/colinc/code/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/annotated_cpu/checkpoints/post_cell_23.pickle

import numpy as np
import cudf
from elastic.core.common.pandas import is_type_styler
is_orig_output_pd = isinstance(orig_output, (pd.Series, pd.DataFrame, pd.Index))
is_opt_output_pd = isinstance(opt_output, (pd.Series, pd.DataFrame, pd.Index))
is_orig_output_array = isinstance(orig_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
is_opt_output_array = isinstance(opt_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
is_orig_output_styler = is_type_styler(type(orig_output))
is_opt_output_styler = is_type_styler(type(opt_output))
if is_orig_output_styler and is_opt_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_orig_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_opt_output_styler:
    assert opt_output.to_html() == orig_output

if is_orig_output_pd and is_opt_output_pd:
    assert orig_output.equals(opt_output)
# TODO(jie): this is a hack.
elif ((is_orig_output_pd or is_opt_output_pd) and (is_orig_output_array or is_opt_output_array)) or (is_orig_output_array and is_opt_output_array):
    assert list(orig_output) == list(opt_output)
else:
    assert orig_output == opt_output


trying: ['title_for_x_axis', 'title_for_x_axis_cell20', 'title_for_x_axis_cell33', 'title_for_x_axis_cell22', 'title_for_x_axis_cell29']
me:  12
me:  16
me:  42
me:  20
me:  34
trying: ['title_for_x_axis', 'title_for_x_axis_cell20', 'title_for_x_axis_cell33', 'title_for_x_axis_cell22', 'title_for_x_axis_cell29']
me:  12
me:  16
me:  42
me:  20
me:  34
trying: ['responses_df_2022_cell10', 'responses_df_2022']


me:  32
me:  32
trying: ['responses_df_2022_cell10', 'responses_df_2022']


me:  32
me:  32
trying: ['question_name', 'question_of_interest_cell18']
me:  12
me:  12
trying: ['question_name', 'question_of_interest_cell18']
me:  12
me:  12
trying: ['title_for_x_axis', 'title_for_x_axis_cell20', 'title_for_x_axis_cell33', 'title_for_x_axis_cell22', 'title_for_x_axis_cell29']
me:  12
me:  16
me:  42
me:  20
me:  34
trying: ['title_for_x_axis', 'title_for_x_axis_cell20', 'title_for_x_axis_cell33', 'title_for_x_axis_cell22', 'title_for_x_axis_cell29']
me:  12
me:  16
me:  42
me:  20
me:  34
trying: ['file_name_2018', 'file_name_2017']
me:  2
me:  2
trying: ['file_name_2018', 'file_name_2017']
me:  2
me:  2
trying: ['file_name_2019']
me:  2
trying: ['orientation_for_chart']
me:  10
trying: ['load_survey_data']
me:  2
trying: ['percentages_cell23']
me:  22
trying: ['question_of_interest_cell23']
me:  22
trying: ['count_then_return_percent_23']
me:  22
trying: ['online_learning_platforms_df_2022']
me:  26
trying: ['question_of_interest_cell25']
me:  26
trying: ['grab_s

me:  34
trying: ['responses_df_2017']


me:  34
trying: ['responses_df_2019_cell10']


...

me:  42
trying: ['count_then_return_percent_33']
me:  42
trying: ['question_of_interest_cell33']
me:  42
trying: ['online_learning_platforms_df_2022_cell27']
me:  30
trying: ['country_df_combined_cell18']
me:  12
trying: ['add_year_column_to_dataframes_33']
me:  42
trying: ['question_of_interest_cell27']
me:  30
trying: ['count_then_return_percent_18']
me:  12
trying: ['grab_subset_of_data_27']
me:  30
trying: ['country_df_combined']
me:  12
trying: ['question_name_alternate']
me:  12
trying: ['add_year_column_to_dataframes_18']
me:  12
trying: ['subset_of_countries']
me:  12
trying: ['question_name_alternate_cell18']
me:  12
trying: ['USA_responses_df_2022']
me:  24
trying: ['combine_subset_of_data_from_multiple_years_18']
me:  12
trying: ['title_for_chart_cell24']
me:  24
trying: ['title_for_y_axis_cell24']
me:  24
trying: ['question_of_interest_cell24']
me:  24
trying: ['bar_chart_multiple_choice_24']
me:  24
trying: ['percentages_cell24']
me:  24
trying: ['count_then_return_percent

me:  24
trying: ['grab_subset_of_data_34']
me:  44
trying: ['online_learning_platforms_df_2022_cell34']
me:  44
trying: ['question_of_interest_cell34']
me:  44
trying: ['pd']
me:  0
trying: ['glob']
me:  0
trying: ['base_dir_2020']
me:  2
trying: ['directory_cell8']
me:  2
trying: ['df_cell35']
me:  46
trying: ['base_dir_2019']
me:  2
trying: ['combine_subset_of_data_from_multiple_years_for_multiple_choice_multiple_response_questions_35']
me:  46
trying: ['factor']
me:  2
trying: ['question_of_interest_cell28']
me:  32
trying: ['base_dir_2022']
me:  2
trying: ['count_then_return_percent_28']
me:  32
trying: ['base_dir_2017']
me:  2
trying: ['percentages_cell28']
me:  32
trying: ['grab_subset_of_data_35']
me:  46
trying: ['responses_in_order_cell28']
me:  32
trying: ['responses_df_2018']


me:  2
trying: ['responses_df_2019']


me:  2
trying: ['age_df_combined_cell22']
me:  20
trying: ['language_df_combined']
me:  46
trying: ['percentages']
me:  10
trying: ['add_year_column_to_dataframes_22']
me:  20
trying: ['language_df_combined_percentages_cell35']
me:  46
trying: ['question_of_interest_cell22']
me:  20
trying: ['question_of_interest_cell35']
me:  46
trying: ['count_then_return_percent_22']
me:  20
trying: ['add_year_column_to_dataframes_35']
me:  46
trying: ['combine_subset_of_data_from_multiple_years_22']
me:  20
trying: ['replace_hyphen_with_en_dash']
me:  4
trying: ['convert_df_of_counts_to_percentages_35']
me:  46
trying: ['language_df_combined_percentages']
me:  46
trying: ['age_df_combined']
me:  16
trying: ['language_df_combined_counts']
me:  46
trying: ['degree_df_combined_cell29']
me:  34
trying: ['subset_of_degrees']
me:  34
trying: ['add_year_column_to_dataframes_29']
me:  34
trying: ['title_for_x_axis_cell29']
me:  34
trying: ['count_then_return_percent_29']
me:  34
trying: ['percentages_per_c

Declaring variable sns
Declaring variable np
Declaring variable go
Declaring variable px
Declaring variable warnings
Declaring variable pd
Declaring variable glob
Declaring variable orig_output
Declaring variable file_name_2018
Declaring variable file_name_2017
Declaring variable file_name_2018
Declaring variable file_name_2017
Declaring variable file_name_2019
Declaring variable load_survey_data
Declaring variable base_dir_2020
Declaring variable directory_cell8
Declaring variable base_dir_2019
Declaring variable factor
Declaring variable base_dir_2022
Declaring variable base_dir_2017
Declaring variable responses_df_2018
Declaring variable responses_df_2019
Declaring variable file_name_2022
Declaring variable file_name_2020
Declaring variable base_dir_2018
Declaring variable file_name_2021
Declaring variable directory
Declaring variable base_dir_2021
Declaring variable replace_hyphen_with_en_dash
Declaring variable percentages_per_country_df
Declaring variable create_dataframe_of_coun